# Session 10: SQL for analysts

Eight tasks and ten queries on the Lakeside Mart database. Write each query in SQL. For three tasks, rebuild the answer in pandas and compare.

**SOLUTION VERSION. Tutor only.** Run the cells from the top. Each cell that starts with '# Your turn' is yours to complete. Run it, then run the check cell after it: the check tells you if you are right and what to fix.

In [ ]:
# Run this cell first. It puts the course data files next to your notebook.
import os, zipfile, urllib.request
DATA_ZIP_URL = ""   # your tutor may give you a link; leave empty to upload data.zip instead
if not os.path.exists("orders.csv"):
    if DATA_ZIP_URL:
        urllib.request.urlretrieve(DATA_ZIP_URL, "data.zip")
    else:
        from google.colab import files
        print("Choose the file data.zip from your computer")
        uploaded = files.upload()
        os.replace(list(uploaded)[0], "data.zip")   # works even if the browser renamed it to data (1).zip
    zipfile.ZipFile("data.zip").extractall(".")
print("Data ready:", sorted(f for f in os.listdir(".") if f.endswith((".csv", ".db"))))

## Task 1. Connect and list the tables
Open `lakeside.db` with `sqlite3` as `con`, then run a query that lists the table names into `tables`. SQLite keeps them in `sqlite_master`. Then count the rows of `orders` with `SELECT COUNT(*)` and store the number in `n_orders`.

In [ ]:
import sqlite3
import pandas as pd

con = sqlite3.connect('lakeside.db')
tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type = 'table'", con)
print(tables)
n_orders = pd.read_sql_query('SELECT COUNT(*) AS n FROM orders', con).n[0]
print(n_orders)

In [ ]:
assert len(tables) == 9, 'There should be nine tables'
assert n_orders == 30229, 'Count every row of orders'
print('Task 1 correct')

## Task 2. WHERE with two tests, then the biggest orders
How many orders were paid by `Card` at branch `B06` (Mbarara)? Store the count in `n_card`. Then list the three biggest orders, `order_id` and `order_value_ugx`, largest first, with `ORDER BY` and `LIMIT`, in `top3`.

In [ ]:
q = """SELECT COUNT(*) AS n FROM orders
       WHERE branch_id = 'B06' AND payment_method = 'Card'"""
n_card = pd.read_sql_query(q, con).n[0]
print(n_card)

q = """SELECT order_id, order_value_ugx FROM orders
       ORDER BY order_value_ugx DESC, order_id LIMIT 3"""
top3 = pd.read_sql_query(q, con)
print(top3)

In [ ]:
assert n_card == 257, 'Use AND between the two tests and quote the text'
assert list(top3.order_id) == ['O014303', 'O029299', 'O003810'], 'Sort by order_value_ugx DESC and keep three'
print('Task 2 correct')

## Task 3. Revenue by region, SQL and pandas
Join `orders` to `branches` and total `order_value_ugx` by `region`, largest first. Then rebuild it in pandas with `merge` and `groupby`. Call the SQL result `by_region` and the pandas result `twin`. The two must agree.

In [ ]:
q = """SELECT b.region, SUM(o.order_value_ugx) AS revenue
       FROM orders AS o
       INNER JOIN branches AS b ON o.branch_id = b.branch_id
       GROUP BY b.region
       ORDER BY revenue DESC"""
by_region = pd.read_sql_query(q, con)

orders = pd.read_sql_query('SELECT * FROM orders', con)
branches = pd.read_sql_query('SELECT * FROM branches', con)
twin = (orders.merge(branches, on='branch_id')
              .groupby('region').order_value_ugx.sum()
              .sort_values(ascending=False))
print(by_region)
print(twin)

In [ ]:
assert by_region.revenue[0] == 679043495 and by_region.region[0] == 'Central', 'Central should lead'
assert list(by_region.revenue) == list(twin.values), 'SQL and pandas disagree'
print('Task 3 correct')

## Task 4. HAVING
Which return `reason` values appear more than 450 times in `returns`? Keep `reason` and the count `n`, largest first, in `common`.

In [ ]:
q = """SELECT reason, COUNT(*) AS n
       FROM returns
       GROUP BY reason
       HAVING COUNT(*) > 450
       ORDER BY n DESC"""
common = pd.read_sql_query(q, con)
common

In [ ]:
assert list(common.reason) == ['Damaged', 'Wrong item', 'Expired'], 'Three reasons pass the HAVING test'
print('Task 4 correct')

## Task 5. CASE, SQL and pandas
Sort orders into `Small` (under 20,000), `Medium` (under 100,000) and `Large`, and count each band in SQL into `bands`. Then rebuild the counts with `pd.cut` into `twin_bands`.

In [ ]:
q = """SELECT CASE WHEN order_value_ugx < 20000 THEN 'Small'
                   WHEN order_value_ugx < 100000 THEN 'Medium'
                   ELSE 'Large' END AS band,
              COUNT(*) AS n
       FROM orders GROUP BY band"""
bands = pd.read_sql_query(q, con).set_index('band').n

cut = pd.cut(orders.order_value_ugx, [-1, 19999, 99999, 10**9],
             labels=['Small', 'Medium', 'Large'])
twin_bands = cut.value_counts()
print(bands)
print(twin_bands)

In [ ]:
assert bands['Medium'] == 14486 and bands['Large'] == 2214, 'Check the cut points'
assert all(bands[k] == twin_bands[k] for k in ['Small', 'Medium', 'Large']), 'SQL and pandas disagree'
print('Task 5 correct')

## Task 6. Subquery, SQL and pandas
How many products cost more than the average `unit_price_ugx`? Use a subquery and store the count in `n_dear`, then check it with the pandas mean into `twin_n`.

In [ ]:
q = """SELECT COUNT(*) AS n FROM products
       WHERE unit_price_ugx > (SELECT AVG(unit_price_ugx) FROM products)"""
n_dear = pd.read_sql_query(q, con).n[0]

products = pd.read_sql_query('SELECT * FROM products', con)
twin_n = (products.unit_price_ugx > products.unit_price_ugx.mean()).sum()
print(n_dear, twin_n)

In [ ]:
assert n_dear == 20, 'The subquery goes in brackets after the greater-than sign'
assert n_dear == twin_n, 'SQL and pandas disagree'
print('Task 6 correct')

## Task 7. The walk-in trap
First count the orders whose `customer_id` is NULL, using `IS NULL`, into `walk_in`. Then count the orders where `customer_id <> 'C0495'` and store that count in `not_c0495`, and see how many rows the test drops.

In [ ]:
walk_in = pd.read_sql_query('SELECT COUNT(*) AS n FROM orders WHERE customer_id IS NULL', con).n[0]
q = "SELECT COUNT(*) AS n FROM orders WHERE customer_id <> 'C0495'"
not_c0495 = pd.read_sql_query(q, con).n[0]
print(walk_in, not_c0495, len(orders) - not_c0495)

In [ ]:
assert walk_in == 9171, 'Use IS NULL, and a plain equals sign will not find NULL'
assert not_c0495 == 21050, 'The test drops every NULL row'
assert len(orders) - not_c0495 == 9179, 'Eight C0495 orders plus the walk-ins'
print('Task 7 correct')

## Task 8. Compare and reflect
Open `orders.csv` in Sheets. Build a PivotTable of orders and revenue by branch and match the Task 3 totals. Write one sentence: which of the three SQL and pandas pairs was easiest to read, and why?